In [59]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from PIL import Image
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import resnet50,ResNet50_Weights

In [60]:
data_dir = Path("images").resolve()
dataset = []

for class_folder in data_dir.iterdir():
    if class_folder.name != 'images' and class_folder.is_dir():
        if class_folder.is_dir():
            label = class_folder.name

            for image_file in class_folder.iterdir():
                if image_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    dataset.append({
                        'image_path': str(image_file),
                        'file_name' : image_file.name,
                        'label': label
                    })

dataset = pd.DataFrame(dataset)
dataset.head()

,image_path,file_name,label
0,C:\Users\Andrei\Desktop\project17\images\induc...,induction-coil001.jpg,induction-coil
1,C:\Users\Andrei\Desktop\project17\images\induc...,induction-coil002.jpg,induction-coil
2,C:\Users\Andrei\Desktop\project17\images\induc...,induction-coil003.jpg,induction-coil
3,C:\Users\Andrei\Desktop\project17\images\induc...,induction-coil004.jpg,induction-coil
4,C:\Users\Andrei\Desktop\project17\images\induc...,induction-coil005.jpg,induction-coil


In [61]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
dataset['label'] = le.fit_transform(dataset['label'])

In [62]:
print(len(set(dataset['label'])))

24


In [63]:
from sklearn.model_selection import train_test_split
datatrain, datatest = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset['label'])
dataval, datatest = train_test_split(datatest, test_size=0.5, random_state=42, stratify=datatest['label'])

In [64]:
img_size = 224

In [65]:
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [66]:
class TrainDataset(Dataset):
  def __init__(self, df, transform=None, path_col='image_path', label_col='label'):
    self.df = df
    self.transform = transform
    self.path_col = df[path_col].values
    self.labels = df[label_col].values
  
  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    path = str(self.path_col[index])
    img = Image.open(path).convert('RGB')
    if self.transform:
      img = self.transform(img)
    return img, torch.tensor(self.labels[index], dtype=torch.long)

class TestDataset(Dataset):
  def __init__(self, df, transform=None, path_col='image_path'):
    self.df = df
    self.transform = transform
    self.path_col = df[path_col].values
  
  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    path = str(self.path_col[index])
    img = Image.open(path).convert('RGB')
    if self.transform:
      img = self.transform(img)
    return img

In [67]:
train_dataset = TrainDataset(
    df=datatrain,
    transform=train_transform,
    label_col='label',
    path_col='image_path'
)

validation_dataset = TrainDataset(
    df=dataval,
    transform=test_transform,
    label_col='label',
    path_col='image_path'
)

test_dataset = TestDataset(
    df=datatest,
    transform=test_transform,
    path_col='image_path'
)

In [68]:
BATCH_SIZE=32
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0
)
validation_loader=DataLoader(
    validation_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=0
)

In [69]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [70]:
class CNN_preantrenat(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes) 
        )

    def forward(self, x):
        return self.model(x)

model=CNN_preantrenat(num_classes=36).to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.0003)

In [71]:
epochs=10
train_losses =[]
val_losses=[]
val_accuracies=[]

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    model.eval()
    running_val_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()
  
            _, predicted = torch.max(outputs.data, 1) 

            
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
      
    avg_val_loss = running_val_loss / len(validation_loader)
    val_losses.append(avg_val_loss)
    
    avg_val_accuracy = correct_predictions / total_samples
    val_accuracies.append(avg_val_accuracy)
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} Validation Loss:{avg_val_loss:.4f} Validation Accuracy: {avg_val_accuracy:.4f}")

Epoch [1/10] Train Loss: 2.7067 Validation Loss:2.2108 Validation Accuracy: 0.3353
Epoch [2/10] Train Loss: 2.0590 Validation Loss:1.8629 Validation Accuracy: 0.4237
Epoch [3/10] Train Loss: 1.7758 Validation Loss:1.6982 Validation Accuracy: 0.4850
Epoch [4/10] Train Loss: 1.5792 Validation Loss:1.7136 Validation Accuracy: 0.4731
Epoch [5/10] Train Loss: 1.4338 Validation Loss:1.6016 Validation Accuracy: 0.5000
Epoch [6/10] Train Loss: 1.2917 Validation Loss:1.6160 Validation Accuracy: 0.5120
Epoch [7/10] Train Loss: 1.1955 Validation Loss:1.6343 Validation Accuracy: 0.5075
Epoch [8/10] Train Loss: 1.0697 Validation Loss:1.7346 Validation Accuracy: 0.4835
Epoch [9/10] Train Loss: 0.9673 Validation Loss:1.9413 Validation Accuracy: 0.4790
Epoch [10/10] Train Loss: 0.8666 Validation Loss:1.7553 Validation Accuracy: 0.5314


In [72]:
model.eval()
test_predictions=[]
test_image_names=[]
test_loader=DataLoader(test_dataset, 
                       batch_size=BATCH_SIZE, 
                       shuffle=False)

In [73]:
with torch.no_grad():
    for images in test_loader:
        images=images.to(device)
        outputs=model(images)
        _,predicted=torch.max(outputs,1)
        test_predictions.extend(predicted.cpu().numpy())


In [77]:
datatest

,image_path,file_name,label
2644,C:\Users\Andrei\Desktop\project17\images\micro...,microprocessor760.jpg,8
5195,C:\Users\Andrei\Desktop\project17\images\semic...,semiconductor-diode215.jpg,17
1247,C:\Users\Andrei\Desktop\project17\images\limit...,limiter-clipper300.jpg,4
4564,C:\Users\Andrei\Desktop\project17\images\semi-...,semi-conductor020.jpg,16
6506,C:\Users\Andrei\Desktop\project17\images\step-...,step-up-transformer132.jpg,22
...,...,...,...
50,C:\Users\Andrei\Desktop\project17\images\induc...,induction-coil051.jpg,0
2835,C:\Users\Andrei\Desktop\project17\images\omni-...,omni-directional-antenna049.jpg,10
382,C:\Users\Andrei\Desktop\project17\images\jumpe...,jumper-cable232.jpg,1
3053,C:\Users\Andrei\Desktop\project17\images\poten...,potential-divider143.jpg,11


In [79]:
test_predictions = le.inverse_transform(test_predictions)

In [81]:
len(test_predictions)

669

In [87]:
datatest = datatest.reset_index(drop=True)
output = []

for index, row in datatest.iterrows():
  output.append({
    'datapointID' : row['file_name'],
    'answer' : test_predictions[index]
  })
output = pd.DataFrame(output)
output

,datapointID,answer
0,microprocessor760.jpg,microprocessor
1,semiconductor-diode215.jpg,omni-directional-antenna
2,limiter-clipper300.jpg,limiter-clipper
3,semi-conductor020.jpg,microprocessor
4,step-up-transformer132.jpg,step-up-transformer
...,...,...
664,induction-coil051.jpg,solenoid
665,omni-directional-antenna049.jpg,omni-directional-antenna
666,jumper-cable232.jpg,jumper-cable
667,potential-divider143.jpg,stabilizer


In [ ]:
output.to_csv('submission.csv', index = False)